### txt -> xlsx

In [ ]:
import pandas as pd

# 파일 읽기
with open('dataset/rating.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()
cnt_fail = 0

data = []
# 헤더 정의 (열 순서)
header = [
    "UserID",
    "products",
    "Categories of the products",
    "rating",
    "helpfulness",
    "time",
    "content of review",
    "userid:helpfulness voting"
]

for line in lines:
    # 각 리뷰는 </endperson>으로 끝남
    if '</endperson>' not in line:
        continue
    # </endperson> 제거 후 양쪽 공백 제거
    line = line.replace('</endperson>', '').strip()
    
    # 처음 7개 필드를 구분하기 위해 "::::"를 최대 7번(split 후 8개 항목) 분리합니다.
    fields = line.split('::::', 7)
    
    # 만약 7개 미만이면 데이터 형식이 잘못된 것으로 건너뜁니다.
    if len(fields) < 7:
        print(f"Skipping invalid line: {line}")
        cnt_fail+=1
        continue

    if len(fields) < 8:
        voting_str = ""
    else:
        # 투표 정보는 ":::”를 기준으로 분리합니다.
        tokens = [token for token in fields[7].split(':::') if token.strip() != '']
        # 각 토큰은 이미 "userid:helpfulness" 형식이므로, 콤마로 연결합니다.
        voting_str = ", ".join(tokens)
    
    row = fields[:7] + [voting_str]
    data.append(row)

# DataFrame으로 변환 후 Excel 파일로 저장
df = pd.DataFrame(data, columns=header)
df.to_excel('rating.xlsx', index=False, engine='openpyxl')

print("cnt_fail",cnt_fail)


cnt_fail 0


### 용량 / 4, 타입명시

In [13]:
import pandas as pd

# 열별로 데이터 타입을 명시적으로 지정
dtypes = {
    'index': 'int64',
    'UserID': 'str',
    'products': 'str',
    'Categories of the products': 'str',
    'rating': 'float64',
    'helpfulness': 'str',
    'time': 'str',
    'content of review': 'str',
    'userid:helpfulness voting': 'str'
}

origin_file = "./rating-csv2.csv"
transform_file = "./rating-csv4.csv"

df = pd.read_csv(origin_file)

# 타입변환환
for col, col_dtype in dtypes.items():
    df[col] = df[col].astype(col_dtype)

#용량//4
df_reduced = df.iloc[:len(df)//2]

df_reduced.to_csv(transform_file, index=False)
output_df = pd.read_csv(transform_file)
print(output_df.dtypes)


C:\Users\yuvnn\AppData\Local\Temp\ipykernel_10796\195529116.py:19: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(origin_file)


index                           int64
UserID                        float64
products                       object
Categories of the products     object
rating                        float64
helpfulness                    object
time                           object
content of review              object
userid:helpfulness voting      object
dtype: object


### csv 파일 확인

In [1]:
import pandas as pd
df = pd.read_csv('./rating_data/rating-csv.csv')
print(len(df))
df.sample(5)

C:\Users\yuvnn\AppData\Local\Temp\ipykernel_11800\1095641452.py:2: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('./rating_data/rating-csv.csv')


303607


,index,UserID,products,Categories of the products,rating,helpfulness,time,content of review,userid:helpfulness voting
205720,205720,6023030,"The Corniche, Doha",Travel,30.0,very helpful,29.01.2009,When I travelled to Thailand with work som...,"6736287:3, 6044343:4, 6044343:4, 5241315:4, 52..."
5360,5360,5001205.0,Member Advice on Termination of Pregnancy,Ciao Caf?,10.0,very helpful,02.07.2001,~ ~ The topic of abortion is one I have be...,"16685:1, 5241292:4, 5241292:4, 5241315:4, 5241..."
216598,216598,5597511,Motorola razr V3i Dolce & Gabbana limited edit...,Telecommunications,30.0,very helpful,17.08.2006,The stylish Motorola V3i would tempt anyon...,"5241315:3, 5241315:3, 5027487:3, 5439885:3, 54..."
273185,273185,19919.0,Millwall,Ciao Caf?,40.0,helpful,27.01.2001,I went to see top of the table Millwall pl...,"5001532:3, 5012156:3, 5012156:3, 17308:3, 1730..."
84671,84671,5337786.0,Thorntons Almond and Nougat Chocolate,Food & Drink,40.0,very helpful,02.10.2003,I was out wondering around town the other ...,"5318908:3, 5318908:3, 5372101:4, 5378253:4, 53..."


In [21]:
import pandas as pd

# CSV 파일 불러오기
a = pd.read_feather("./output/UR_output/Internet_UR.feather")
b = pd.read_feather("./dataset/rating_data/rating_groupby_category&userID_feather/Internet.feather")
b['UserID'] = b['UserID'].astype(str).apply(lambda x: x.split('.')[0] if '.' in x else x)
a['UserID'] = a['UserID'].astype(str).apply(lambda x: x.split('.')[0] if '.' in x else x)
b
# # 차집합 구하기
# unique_users = set(a['UserID']) - set(b['UserID'])

# # 출력
# print(len(unique_users))

,UserID,content of review
0,23,Online betting sites have thrived recently; al...
1,24,"Oh so fast, so very fast I can t take it the s..."
2,30,A few weeks ago I decided to buy a printer. Fr...
3,31,I discovered ciao whilst looking up a classica...
4,40,I just thought that I would write an opinion t...
...,...,...
4135,6893624,quot;OnePollCashBack.com quot; www.onepollcash...
4136,6893925,"Through school, university and now the Royal M..."
4137,6895524,Thinking about what I fancied reviewing I real...
4138,6896960,History Of The Store. Topman was born in 1978 ...


### (사용x)UserID_Category 로 이루어진 key생성 (output : rating_with_keys.csv)

In [29]:
import pandas as pd
import itertools

input_csv = "./fields.csv"
output_csv = "./rating_with_keys.csv"

df = pd.read_csv(input_csv)

# 고유값 추출, category null제거거
unique_users = df["UserID"].unique()
unique_categories = df["Categories of the products"].str.strip().dropna().unique()
print(f"중복 제거 후 UserID 개수: ",len(unique_users))
print(f"중복 제거 후 Categories of the products 개수: ",len(unique_categories))

# 조합 생성
combinations = list(itertools.product(unique_users, unique_categories))
df_combined = pd.DataFrame(combinations, columns=["UserID", "Categories of the products"])
print(f"생성된 조합 개수: {len(df_combined)}")

# 새로운 key 열 추가 (형식: UserID_Category)
df_combined["key"] = df_combined["UserID"].astype(str) + "_" + df_combined["Categories of the products"].astype(str)

# 열 순서 변경 및 저장
df_combined = df_combined[["key", "UserID", "Categories of the products"]]
df_combined.to_csv(output_csv, index=False)
print(output_csv," 파일 생성 완료!")

df = pd.read_csv(output_csv)
df.head(3)


중복 제거 후 UserID 개수:  10978
중복 제거 후 Categories of the products 개수:  28
생성된 조합 개수: 307384
./rating_with_keys.csv  파일 생성 완료!


C:\Users\yuvnn\AppData\Local\Temp\ipykernel_3288\357828780.py:28: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(output_csv)


,key,UserID,Categories of the products
0,5247778_House & Garden,5247778,House & Garden
1,5247778_Travel,5247778,Travel
2,5247778_Food & Drink,5247778,Food & Drink


In [ ]:
import pandas as pd

# 필드명
review = "content of review"
category = "Categories of the products"

# 파일 경로
origin_csv = "./rating-csv.csv"
key_csv = "./rating_with_keys.csv"
origin_df = pd.read_csv(origin_csv)
key_df = pd.read_csv(key_csv)

# 1. a.csv에서 UserID와 Category 결합하여 user_category 열 만들기
origin_df["user_category"] = origin_df["UserID"].astype(str) + "_" + origin_df[category].astype(str)

# 2. b.csv의 key와 a.csv의 user_category를 병합하여 review를 추가할 준비
# key_df의 key와 origin_df의 user_category가 일치하는 경우 병합
merged_df = pd.merge(key_df, origin_df[['user_category', review]], left_on='key', right_on='user_category', how='left')

# 3. review가 일치할 때, 기존 review에 덧붙이기
# groupby 후, 리뷰 합치기 (중복된 key에 대해 리뷰 덧붙이기)
merged_df['reviews'] = merged_df.groupby('key')[review].transform(lambda x: ' '.join(x.dropna()))

# 4. 병합된 DataFrame에서 중복된 리뷰 제거
merged_df = merged_df.drop_duplicates(subset='key', keep='first')

# 5. review_count 열 계산
merged_df['review_count'] = merged_df.groupby('key')['key'].transform('count')

# 6. 필요한 열만 남기고, key_csv에 반영
key_df = merged_df[['key', 'reviews', 'review_count']]

# 7. 최종 결과 저장
key_df.to_csv(key_csv, index=False)

print("처리가 완료되었습니다.")


C:\Users\yuvnn\AppData\Local\Temp\ipykernel_3288\1294637057.py:10: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  origin_df = pd.read_csv(origin_csv)
C:\Users\yuvnn\AppData\Local\Temp\ipykernel_3288\1294637057.py:11: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  key_df = pd.read_csv(key_csv)


처리가 완료되었습니다.


### (사용x)유저당 카테고리별 리뷰 merge

In [35]:
import pandas as pd

# 필드명
review = "content of review"
category = "Categories of the products"

# 파일 경로
origin_csv = "./rating-csv.csv"
key_csv = "./rating_with_keys.csv"

origin_df = pd.read_csv(origin_csv)
key_df = pd.read_csv(key_csv)

# 1. a.csv에서 UserID와 Category 결합하여 user_category 열 만들기
origin_df["user_category"] = origin_df["UserID"].astype(str) + "_" + origin_df[category].astype(str)

# 2. b.csv의 key와 a.csv의 user_category를 병합하여 review를 추가할 준비
merged_df = pd.merge(key_df, origin_df[['user_category', review]], left_on='key', right_on='user_category', how='left')

# 3. 동일한 key에 대해 review 합치기
# 같은 key에 대해 review를 이어붙임 (중복된 리뷰가 있을 때)
merged_df['reviews'] = merged_df.groupby('key')[review].transform(lambda x: ' '.join(x.dropna()))

# 4. 동일한 key의 중복된 리뷰 제거 (하나만 남기기)
merged_df = merged_df.drop_duplicates(subset='key', keep='first')

# 5. review_count 열 계산 (각 key에 대한 리뷰 개수 계산)
merged_df['review_count'] = merged_df.groupby('key')['key'].transform('count')

# 6. 병합되지 않은 값(merge되지 않은 key)을 계산
unmerged_count = merged_df[review].isna().sum()

# 7. 필요한 열만 남기고, key_csv에 반영
key_df = merged_df[['key', 'reviews', 'review_count']]

# 8. 최종 결과 저장
key_df.to_csv(key_csv, index=False)

# 병합되지 않은 값 출력
print(f"병합되지 않은 값의 개수: {unmerged_count}")

print("처리가 완료되었습니다.")


C:\Users\yuvnn\AppData\Local\Temp\ipykernel_3288\1459132212.py:11: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  origin_df = pd.read_csv(origin_csv)
C:\Users\yuvnn\AppData\Local\Temp\ipykernel_3288\1459132212.py:12: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  key_df = pd.read_csv(key_csv)


병합되지 않은 값의 개수: 288518
처리가 완료되었습니다.


### csv to feather

In [7]:
import pandas as pd
import os

# 입력 및 출력 폴더 지정
input_folder = "./dataset/rating_data/rating_groupby_category&userID"
output_folder = "./dataset/rating_data/rating_groupby_category&userID_feather"

# 출력 폴더가 없으면 생성
os.makedirs(output_folder, exist_ok=True)

# 폴더 내 모든 CSV 파일 처리
for filename in os.listdir(input_folder):
    if filename.endswith(".csv"):  # CSV 파일만 처리
        input_path = os.path.join(input_folder, filename)
        output_path = os.path.join(output_folder, filename.replace(".csv", ".feather"))  # Feather 확장자로 변경

        # 읽기
        df = pd.read_csv(input_path)

        # Feather로 저장
        df.to_feather(output_path)

        print(f"Processed: {filename} -> Saved as Feather to {output_path}")


Processed: Adult Products.csv -> Saved as Feather to ./dataset/rating_data/rating_groupby_category&userID_feather\Adult Products.feather
Processed: Beauty.csv -> Saved as Feather to ./dataset/rating_data/rating_groupby_category&userID_feather\Beauty.feather
Processed: Books.csv -> Saved as Feather to ./dataset/rating_data/rating_groupby_category&userID_feather\Books.feather
Processed: Cameras.csv -> Saved as Feather to ./dataset/rating_data/rating_groupby_category&userID_feather\Cameras.feather
Processed: Cars & Motorcycles.csv -> Saved as Feather to ./dataset/rating_data/rating_groupby_category&userID_feather\Cars & Motorcycles.feather
Processed: Ciao Caf_.csv -> Saved as Feather to ./dataset/rating_data/rating_groupby_category&userID_feather\Ciao Caf_.feather
Processed: Computers_cu.csv -> Saved as Feather to ./dataset/rating_data/rating_groupby_category&userID_feather\Computers_cu.feather
Processed: DVDs.csv -> Saved as Feather to ./dataset/rating_data/rating_groupby_category&userID

# URE

### category별로 grouping (ouput : csv)

In [3]:
import pandas as pd
import re

# CSV 파일 불러오기
df = pd.read_csv("./rating-csv.csv")

# 파일명으로 사용할 수 없는 문자 정의
def safe_filename(category):
    clean_name = re.sub(r'[\/:*?"<>|]', '_', category).strip()
    return clean_name if clean_name else None

# unname 파일 카운트
unname_count = 1

# category 별로 파일 저장
for category, group in df.groupby("Categories of the products"):
    valid_name = safe_filename(category)
    if not valid_name:  # 파일명으로 사용할 수 없으면 unname으로 저장
        filename = f"rating_unname_{unname_count}.csv"
        unname_count += 1
    else:
        filename = f"rating_{valid_name}.csv"

    group.to_csv(filename, index=False)
    print(f"Saved {filename}")


C:\Users\yuvnn\AppData\Local\Temp\ipykernel_1228\3896010267.py:5: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("./rating-csv.csv")


Saved rating_Adult Products.csv
Saved rating_Beauty.csv
Saved rating_Books.csv
Saved rating_Cameras.csv
Saved rating_Cars & Motorcycles.csv
Saved rating_Ciao Caf_.csv
Saved rating_Computers.csv
Saved rating_DVDs.csv
Saved rating_Education & Careers.csv
Saved rating_Electronics.csv
Saved rating_Entertainment.csv
Saved rating_Family.csv
Saved rating_Fashion.csv
Saved rating_Finance.csv
Saved rating_Food & Drink.csv
Saved rating_Games.csv
Saved rating_Health.csv
Saved rating_House & Garden.csv
Saved rating_Household Appliances.csv
Saved rating_Internet.csv
Saved rating_Music.csv
Saved rating_Musical Instruments & Equipment.csv
Saved rating_Office Equipment.csv
Saved rating_Shopping.csv
Saved rating_Software.csv
Saved rating_Sports & Outdoors.csv
Saved rating_Telecommunications.csv
Saved rating_Travel.csv


### userID별로 grouping

In [8]:
import pandas as pd

# 파일 경로
file_path = "./rating_data/rating_groupby_category/rating_Books.csv"

# CSV 파일 읽기
df = pd.read_csv(file_path)

# 데이터 샘플 확인
print(df.info())  # 데이터 타입 및 결측치 확인


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23883 entries, 0 to 23882
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   index                       23883 non-null  int64  
 1   UserID                      23883 non-null  float64
 2   products                    23883 non-null  object 
 3   Categories of the products  23883 non-null  object 
 4   rating                      23883 non-null  float64
 5   helpfulness                 23883 non-null  object 
 6   time                        23883 non-null  object 
 7   content of review           23883 non-null  object 
 8   userid:helpfulness voting   23850 non-null  object 
dtypes: float64(2), int64(1), object(6)
memory usage: 1.6+ MB
None


In [16]:
import pandas as pd
import os

# 입력 및 출력 폴더 지정
input_folder = "./rating_data/rating_groupby_category"
output_folder = "./rating_data/rating_groupby_category&userID"

# 출력 폴더가 없으면 생성
os.makedirs(output_folder, exist_ok=True)

# a 폴더 내 모든 CSV 파일 처리
for filename in os.listdir(input_folder):
    if filename.endswith(".csv"):  # CSV 파일만 처리
        input_path = os.path.join(input_folder, filename)
        output_path = os.path.join(output_folder, filename)  # 같은 이름으로 저장

        # 파일 읽기 (UTF-8로 시도하고 실패하면 다른 인코딩 사용)
        try:
            df = pd.read_csv(input_path, encoding="utf-8-sig")  # utf-8-sig로 먼저 시도
        except UnicodeDecodeError:
            df = pd.read_csv(input_path, encoding="ISO-8859-1")  # 실패 시 ISO-8859-1로 읽기

        # NaN 값이 있는 행 제거
        df = df.dropna(subset=["UserID", "content of review"])

        # 데이터 타입 확인 (문자열로 변환)
        df["content of review"] = df["content of review"].astype(str)

        # 개행 문자 및 공백 정리 후 userID별로 리뷰 결합
        df2 = df.groupby("UserID", as_index=False)["content of review"].apply(
            lambda x: " ".join(x.str.replace(r'\s+', ' ', regex=True).str.strip())
        ).reset_index(drop=True)

        # 저장
        df2.to_csv(output_path, index=False, encoding="utf-8-sig")

        print(f"Processed: {filename} -> Saved to {output_path}")


Processed: rating_Adult Products.csv -> Saved to ./rating_data/rating_groupby_category&userID\rating_Adult Products.csv
Processed: rating_Beauty.csv -> Saved to ./rating_data/rating_groupby_category&userID\rating_Beauty.csv
Processed: rating_Books.csv -> Saved to ./rating_data/rating_groupby_category&userID\rating_Books.csv
Processed: rating_Cameras.csv -> Saved to ./rating_data/rating_groupby_category&userID\rating_Cameras.csv
Processed: rating_Cars & Motorcycles.csv -> Saved to ./rating_data/rating_groupby_category&userID\rating_Cars & Motorcycles.csv
Processed: rating_Ciao Caf_.csv -> Saved to ./rating_data/rating_groupby_category&userID\rating_Ciao Caf_.csv
Processed: rating_Computers.csv -> Saved to ./rating_data/rating_groupby_category&userID\rating_Computers.csv
Processed: rating_DVDs.csv -> Saved to ./rating_data/rating_groupby_category&userID\rating_DVDs.csv
Processed: rating_Education & Careers.csv -> Saved to ./rating_data/rating_groupby_category&userID\rating_Education & Ca

In [33]:
import pandas as pd
import os
import re

# 원본 폴더 경로와 새로 저장할 폴더 경로 설정
input_folder_path = "./rating_data/rating_groupby_category&userID"
output_folder_path = "./rating_data/rating_groupby_category&userID_clean"

# 새로 저장할 폴더가 없으면 생성
if not os.path.exists(output_folder_path):
    os.makedirs(output_folder_path)

# 원본 폴더 내 모든 CSV 파일 읽기
for filename in os.listdir(input_folder_path):
    if filename.endswith(".csv"):
        input_file_path = os.path.join(input_folder_path, filename)
        
        # CSV 파일 읽기
        df = pd.read_csv(input_file_path)

        # 'UserID'가 숫자가 아닌 값을 가진 행을 제거
        df['UserID'] = df['UserID'].astype(str).apply(lambda x: x.split('.')[0])
        df = df[df['UserID'].apply(lambda x: bool(re.match(r'^\d+$', str(x).strip())))]

        # 새로 저장할 폴더에 처리된 데이터를 새로운 CSV로 저장
        output_file_path = os.path.join(output_folder_path, filename)
        df.to_csv(output_file_path, index=False)

        print(f"Processed and saved: {filename}")


Processed and saved: rating_Adult Products.csv
Processed and saved: rating_Beauty.csv
Processed and saved: rating_Books.csv
Processed and saved: rating_Cameras.csv
Processed and saved: rating_Cars & Motorcycles.csv
Processed and saved: rating_Ciao Caf_.csv
Processed and saved: rating_Computers.csv
Processed and saved: rating_DVDs.csv
Processed and saved: rating_Education & Careers.csv
Processed and saved: rating_Electronics.csv
Processed and saved: rating_Entertainment.csv
Processed and saved: rating_Family.csv
Processed and saved: rating_Fashion.csv
Processed and saved: rating_Finance.csv
Processed and saved: rating_Food & Drink.csv
Processed and saved: rating_Games.csv
Processed and saved: rating_Health.csv
Processed and saved: rating_House & Garden.csv


KeyboardInterrupt: 

In [9]:
import os
import pandas as pd

# a 폴더 경로 설정
folder_path = "./dataset/rating_data/rating_groupby_category&userID_feather"

# 폴더 내 모든 파일 가져오기
files = [f for f in os.listdir(folder_path) if f.endswith('.feather')]

# 각 CSV 파일에 대해 행 수 출력
total = 0
for file in files:
    file_path = os.path.join(folder_path, file)
    df = pd.read_feather(file_path)
    total += len(df)
    print(f'{file} : {len(df)} ')

print('total : ',total)


Adult Products.feather : 49 
Beauty.feather : 3714 
Books.feather : 3720 
Cameras.feather : 1775 
Cars & Motorcycles.feather : 1914 
Ciao Caf_.feather : 5110 
Computers.feather : 2560 
Computers_cu.feather : 2560 
DVDs.feather : 5047 
Education & Careers.feather : 1059 
Electronics.feather : 2317 
Entertainment.feather : 2677 
Family.feather : 1822 
Fashion.feather : 105 
Finance.feather : 1469 
Food & Drink.feather : 3351 
Games.feather : 3564 
Health.feather : 1918 
House & Garden.feather : 1865 
Household Appliances.feather : 2297 
Internet.feather : 4140 
Music.feather : 3717 
Musical Instruments & Equipment.feather : 265 
Office Equipment.feather : 347 
Shopping.feather : 2599 
Software.feather : 1247 
Sports & Outdoors.feather : 703 
Telecommunications.feather : 3388 
Travel.feather : 4138 
total :  69437


### merge

In [1]:
import os
import pandas as pd

# 입력 폴더와 출력 파일 경로 지정
input_folder = "./rating_data/rating_groupby_category&userID"
output_file = "./rating_data/rating_merged.feather"

# CSV 파일들을 담을 리스트
dfs = []

for filename in os.listdir(input_folder):
    if filename.endswith(".csv"):
        filepath = os.path.join(input_folder, filename)
        df = pd.read_csv(filepath)

        df["UserID"] = df["UserID"].astype(str) #float오류 제거

        category = filename.replace("rating_", "").replace(".csv", "")
        df["ID"] = df["UserID"].astype(str) + "_" + category
        df = df[["ID", "content of review"]]
        dfs.append(df)

# 모든 데이터프레임을 병합
merged_df = pd.concat(dfs, ignore_index=True)
merged_df.to_feather(output_file) #csv는 용량 너무 커서 feather로 저장

print(f"Merged feather saved to {output_file}")
df = pd.read_feather(output_file)

print(df.head(10))

Merged feather saved to ./rating_data/rating_merged.feather
                         ID                                  content of review
0     2815.0_Adult Products      Talk Dirty to Me is brilliant. When you ar...
1    11996.0_Adult Products      I have had this game for about 15 years an...
2  5001156.0_Adult Products      I love K.Y Jelly. It is one of those produ...
3  5002475.0_Adult Products      The battle of the sexes. The ultimate game...
4  5016738.0_Adult Products      I suppose the only way to tackle an opinio...
5  5017708.0_Adult Products      As it s after the  watershed , I thought I...
6  5035572.0_Adult Products      Once a month, if you are having periods, y...
7  5035648.0_Adult Products      Now, before you all jump to the conclusion...
8  5046112.0_Adult Products      This game is perfect for parties  It s not...
9  5050984.0_Adult Products      The Battle of the Sexes - it says it all r...


# UI

### userID, time, review

In [3]:
import csv

origin_csv = "./rating_data/rating-csv.csv"
output_csv = "./rating_data/rating_for_UI.csv"

# 원본 파일 열기 & 새 파일에 저장
with open(origin_csv, mode="r", encoding="utf-8") as infile, open(output_csv, mode="w", encoding="utf-8", newline="") as outfile:
    reader = csv.DictReader(infile)  # CSV를 딕셔너리 형태로 읽기
    fieldnames = ['UserID', 'time','content of review']  # 필요한 열만 선택
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)

    writer.writeheader()  # 헤더 작성
    for row in reader:
        writer.writerow({field: row[field] for field in fieldnames})


import pandas as pd
df = pd.read_csv(output_csv)
df.sample(5)


C:\Users\yuvnn\AppData\Local\Temp\ipykernel_11800\884003457.py:18: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(output_csv)


,UserID,time,content of review
141432,6152305.0,31.10.2007,***Jerome Russell B blonde Highlighting Ki...
152473,5080915.0,29.06.2002,The Aquafresh toothpaste I am going to wri...
238473,5421792,16.08.2004,LIKE MOST WOMEN WHO TRY TO LOOK AFTER THEI...
264466,5096023.0,21.01.2002,As our Buzz aeroplane descended through th...
182472,5693034.0,24.08.2006,Some mothers do ave em is a British come...


In [8]:
import pandas as pd
from datetime import datetime

df = pd.read_csv(output_csv)

df['UserID'] = df['UserID'].astype(str)
df['UserID'] = df['UserID'].apply(lambda x: x.split('.')[0])

# 날짜 변환 (DD.MM.YYYY → timestamp)
df['time'] = pd.to_datetime(df['time'], format='%d.%m.%Y')
df['time'] = df['time'].astype('int64') // 10**9  # 초 단위로 변환

output_file = "./rating_data/rating_for_UI.feather"
df.to_feather(output_file) 

df = pd.read_feather(output_file)
df.sample(5)

C:\Users\yuvnn\AppData\Local\Temp\ipykernel_11800\2430417311.py:4: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(output_csv)


,UserID,time,content of review
119393,23007,985910400,"The tube, silver and shinyThe paste, white..."
88449,5271854,1052956800,"Hi guys.SueMagee,where are you?I have some..."
23122,5095882,1005609600,I was needing a fitness machine for my fit...
249977,4454,978048000,I got the MZR70 (Silver) as a christmas pr...
100883,5217069,1056240000,For no particular reason I thought I would...
